# 02. 카운트 상태머신 + 전체 데이터 검증

1. `pitch_result`가 비어 있는 행 확인
2. `apply_pitch()` 정의 + 간단한 테스트
3. 모든 타석에 대해 "이전 카운트 + 투구 결과 = 다음 행 카운트" 검증
4. 상태머신이 판단한 타석 종료 방식 vs 데이터의 `events` 비교

In [7]:
import pandas as pd

SEASON = 2023  # 2023~2026
URL = f"https://huggingface.co/datasets/slothman3878/kbo_playbyplay/resolve/main/v0/kbo_pbp_{SEASON}.parquet"
df = pd.read_parquet(URL)
print(SEASON, df.shape)

2023 (219934, 48)


In [8]:
# pitch_result가 None인 51행은 무엇인가? (고의4구 = pitch_number 0 으로 예상)
no_result = df[df["pitch_result"].isna()]
print(no_result["pitch_number"].value_counts(), "\n")
print(no_result["events"].value_counts(dropna=False))
no_result[["game_pk", "inning", "at_bat_number", "pitch_number", "balls", "strikes", "events"]].head(10)

pitch_number
0    95
Name: count, dtype: int64 

events
walk    95
Name: count, dtype: int64


,game_pk,inning,at_bat_number,pitch_number,balls,strikes,events
342,20230401HHWO02023,10,81,0,0,0,walk
1295,20230401LTOB02023,9,86,0,0,0,walk
6480,20230407OBHT02023,8,56,0,0,0,walk
7385,20230407WONC02023,8,54,0,0,0,walk
12095,20230412HHHT02023,8,58,0,0,0,walk
14946,20230414HHKT02023,8,75,0,0,0,walk
21764,20230419OBHH02023,9,74,0,0,0,walk
22712,20230420HTLT02023,7,63,0,0,0,walk
27853,20230423LTNC02023,9,71,0,0,0,walk
28472,20230423WOSK02023,8,80,0,0,0,walk


In [9]:
STRIKE_CODES = {"T", "S", "V", "W"}  # 스트라이크, 헛스윙, 번트헛스윙, 번트파울(2S 이후에도 스트라이크)


def apply_pitch(balls, strikes, result):
    """투구 하나를 반영해서 (balls, strikes, 타석종료여부) 반환"""
    if result == "B":
        balls += 1
        return balls, strikes, balls == 4
    if result in STRIKE_CODES:
        strikes += 1
        return balls, strikes, strikes == 3
    if result == "F":  # 일반 파울은 2스트라이크에서 멈춤
        return balls, min(strikes + 1, 2), False
    if result == "H":  # 인플레이 타구
        return balls, strikes, True
    raise ValueError(f"알 수 없는 투구 결과: {result}")


# 01에서 본 실제 사례로 테스트
assert apply_pitch(3, 2, "F") == (3, 2, False)  # 풀카운트 파울 → 유지
assert apply_pitch(3, 2, "B") == (4, 2, True)   # 볼넷
assert apply_pitch(2, 2, "S") == (2, 3, True)   # 헛스윙 삼진
assert apply_pitch(0, 2, "W") == (0, 3, True)   # 2S 번트파울 → 삼진
assert apply_pitch(1, 0, "F") == (1, 1, False)
assert apply_pitch(2, 0, "H") == (2, 0, True)
print("테스트 통과")

테스트 통과


In [10]:
def end_kind(balls, strikes, last_result):
    if balls == 4:
        return "walk"
    if strikes == 3:
        return "strikeout"
    if last_result == "H":
        return "in_play"
    return "open"  # 카운트상으로는 타석이 안 끝남 (사구, 주루사로 이닝 종료 등)


pitches = df[df["pitch_result"].notna()]
issues = []
pa_rows = []

for (game, ab), pa in pitches.groupby(["game_pk", "at_bat_number"], sort=False):
    balls, strikes, over = 0, 0, False
    for r in pa.itertuples(index=False):
        where = (game, ab, r.pitch_number)
        if over:
            issues.append(("pitch_after_end", *where, ""))
        if (r.balls, r.strikes) != (balls, strikes):
            issues.append(("count_mismatch", *where, f"예상 {balls}-{strikes}, 데이터 {r.balls}-{r.strikes}"))
            balls, strikes = r.balls, r.strikes  # 데이터 값으로 다시 맞추고 계속
        balls, strikes, over = apply_pitch(balls, strikes, r.pitch_result)
    pa_rows.append((game, ab, r.events, end_kind(balls, strikes, r.pitch_result)))

issues = pd.DataFrame(issues, columns=["kind", "game_pk", "at_bat_number", "pitch_number", "detail"])
pa_end = pd.DataFrame(pa_rows, columns=["game_pk", "at_bat_number", "events", "end_kind"])

print("검사한 투구:", len(pitches), "/ 타석:", len(pa_end))
print(issues["kind"].value_counts())

검사한 투구: 219839 / 타석: 56306
kind
count_mismatch    14
Name: count, dtype: int64


In [11]:
# 상태머신이 본 타석 종료 방식 vs 데이터의 타석 결과
pd.crosstab(pa_end["events"].fillna("None"), pa_end["end_kind"], margins=True)

end_kind,in_play,open,strikeout,walk,All
events,,,,,
None,0,170,0,0,170
catcher_interference,1,0,0,0,1
double,2219,0,0,0,2219
double_play,1015,0,0,0,1015
field_error,537,0,0,0,537
field_out,23074,0,2,0,23076
fielders_choice,1674,0,0,0,1674
hit_by_pitch,0,636,0,60,696
home_run,924,0,0,0,924


In [12]:
# 불일치 사례 종류별로 몇 개씩
issues.groupby("kind").head(5)

,kind,game_pk,at_bat_number,pitch_number,detail
0,count_mismatch,20230827WOSS02023,1,1,"예상 0-0, 데이터 0-1"
1,count_mismatch,20230827WOSS02023,1,2,"예상 0-2, 데이터 0-1"
2,count_mismatch,20230913LTHT02023,1,1,"예상 0-0, 데이터 0-1"
3,count_mismatch,20230913LTHT02023,1,2,"예상 0-2, 데이터 0-1"
4,count_mismatch,20230915LGHH02023,1,1,"예상 0-0, 데이터 1-0"
